In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
!update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java
!java -version

In [ ]:
!pip install --force-reinstall pyspark==3.4
!pip install findspark

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.mllib.random import RandomRDDs
from pyspark.sql.types import*

spark = SparkSession.builder.appName("Top5Devices").getOrCreate()
sc = spark.sparkContext
data = sc.textFile("view_data_lab3.csv")
header = data.first()
data_no_header = data.filter(lambda line: line != header)

def parse_line(line):
    x = line.split(',')
    if len(x) == 5:
        event_time = x[0]
        station_num = x[1]
        mso_code = x[2]
        device_id = x[3]
        event_date = x[4]
        return (event_time, station_num, mso_code, device_id, event_date)
    else:
        return None

split_data = data_no_header.map(parse_line).filter(lambda x: x is not None)

def is_prime_time(event_time):
    try:
        hour = int(event_time)
        return 200000 <= hour < 230000
    except ValueError:
        return False

# Find Num Of Dates
unique_dates = split_data.map(lambda row: row[4]).distinct().collect()
num_dates = len(unique_dates)

prime_time_data = split_data.filter(lambda x: is_prime_time(x[0]))

# Map to (device_id, event_date) -> 1
device_date_pairs = prime_time_data.map(lambda x: ((x[3], x[4]), 1))

device_date_counts = device_date_pairs.reduceByKey(lambda a, b: a + b)

# Map to device_id -> (total_events)
device_totals = device_date_counts.map(lambda x: (x[0][0], (x[1])))

device_sums = device_totals.reduceByKey(lambda a,b: a + b)

# Calculate average events per device
device_averages = device_sums.map(lambda x: (x[0], x[1] / num_dates))

# Get top 5 devices with the highest averages
top_5_devices = device_averages.takeOrdered(5, key=lambda x: -x[1])

print("Top 5 devices with the highest average viewing events during Prime-Time:")
for device, avg in top_5_devices:
    print(f"{device} \t {avg:.4f}")

spark.stop()
